# Deployed Retention Policy Fairness

This notebook audits subgroup allocation under the exact frozen budget-constrained expected-value policy. It distinguishes that policy from the probability-only top-10% model-ranking diagnostic.

All employees, outcomes, salaries, and results are synthetic. The evidence is descriptive, noncausal, aggregate only, and cannot authorize automated employment action.

In [1]:
from pathlib import Path

import pandas as pd

PROJECT_ROOT = Path.cwd().resolve().parent if Path.cwd().name == "notebooks" else Path.cwd().resolve()
OUTPUT = PROJECT_ROOT / "data" / "processed" / "retention_policy_fairness"

overall = pd.read_csv(OUTPUT / "selection_rule_overall.csv")
groups = pd.read_csv(OUTPUT / "deployed_policy_group_metrics.csv")
comparison = pd.read_csv(OUTPUT / "selection_rule_comparison.csv")
disparities = pd.read_csv(OUTPUT / "deployed_policy_disparities.csv")
checks = pd.read_csv(OUTPUT / "validation_checks.csv")

## Exact rule comparison

The two rows below use the same calibrated probabilities but different selection objectives. The expected-value policy also uses salary-based replacement cost, budget, capacity, and positive-value constraints.

In [2]:
overall.round(4)

,selection_rule,selected_count,selection_rate,selected_positive_cases,precision,capture_rate,mean_selected_probability,mean_selected_salary_usd,predicted_net_value_usd,cross_rule_overlap_count,cross_rule_jaccard
0,Frozen budget-constrained expected value,700,0.1268,85,0.1214,0.1451,0.1291,121964.4286,840973.8747,184,0.1721
1,Probability-only top 10% diagnostic,553,0.1002,109,0.1971,0.1860,0.1678,70136.3472,236302.6829,184,0.1721


## Corrected headline subgroup rates

These comparisons show why a probability-only proxy cannot be presented as deployed-policy fairness.

In [3]:
headline = comparison.loc[
    ((comparison["attribute"] == "department_name") & comparison["group"].isin(["Customer Support", "Engineering"]))
    | ((comparison["attribute"] == "employment_type") & comparison["group"].isin(["Hourly", "Salaried"]))
    | ((comparison["attribute"] == "job_level") & comparison["group"].isin(["Level 1", "Level 3"]))
]
headline[[
    "attribute",
    "group",
    "sample_size",
    "probability_proxy_selection_rate",
    "deployed_selection_rate",
    "selection_rate_difference_deployed_minus_proxy",
]].round(4)

,attribute,group,sample_size,probability_proxy_selection_rate,deployed_selection_rate,selection_rate_difference_deployed_minus_proxy
14,employment_type,Hourly,889,0.2643,0.0067,-0.2576
15,employment_type,Salaried,4632,0.0687,0.1498,0.0812
16,department_name,Customer Support,399,0.4461,0.0125,-0.4336
17,department_name,Engineering,1153,0.0121,0.2402,0.2281
24,job_level,Level 1,954,0.2516,0.0063,-0.2453
26,job_level,Level 3,1838,0.0277,0.1921,0.1643


## Deployed-policy subgroup metrics

Outcome metrics use the 2024 validation period only. The reserved final-test target is not read.

In [4]:
groups[[
    "attribute",
    "group",
    "sample_size",
    "selected_count",
    "selection_rate",
    "true_positive_rate",
    "false_positive_rate",
    "precision",
    "selected_mean_salary_usd",
    "eligible_for_disparity_comparison",
]].round(4)

,attribute,group,sample_size,selected_count,selection_rate,true_positive_rate,false_positive_rate,precision,selected_mean_salary_usd,eligible_for_disparity_comparison
0,age_band,30-39,1363,174,0.1277,0.1111,0.1295,0.0862,125650.0000,True
1,age_band,40-49,1405,194,0.1381,0.1538,0.1361,0.1237,121440.2062,True
2,age_band,50-59,1279,157,0.1228,0.1654,0.1178,0.1401,123148.4076,True
3,age_band,60+,335,26,0.0776,0.0909,0.0762,0.1154,104703.8462,True
4,age_band,Under 30,1139,149,0.1308,0.1628,0.1267,0.1409,120107.3826,True
5,education_level,Associate,807,47,0.0582,0.0400,0.0601,0.0638,117819.1489,True
6,education_level,Bachelor's,2593,348,0.1342,0.1434,0.1331,0.1178,120202.2989,True
7,education_level,Doctorate,176,7,0.0398,0.0000,0.0424,0.0000,144842.8571,False
8,education_level,High School,689,24,0.0348,0.0549,0.0318,0.2083,94079.1667,True
9,education_level,Master's,1256,274,0.2182,0.2927,0.2101,0.1314,126771.5328,True


## Descriptive disparity summary

Review flags are portfolio screening devices, not legal thresholds or binary fairness conclusions.

In [5]:
disparities.round(4)

,attribute,eligible_groups,selection_rate_difference,lowest_selection_group,highest_selection_group,true_positive_rate_difference,lowest_true_positive_rate_group,highest_true_positive_rate_group,false_positive_rate_difference,lowest_false_positive_rate_group,highest_false_positive_rate_group,screening_review_flag,review_triggers
0,age_band,5,0.0605,60+,40-49,0.0745,60+,50-59,0.0599,60+,40-49,False,NaN
1,education_level,4,0.1833,High School,Master's,0.2527,Associate,Master's,0.1783,High School,Master's,True,"selection_rate, true_positive_rate, false_posi..."
2,region,4,0.1184,South,Northeast,0.0886,South,Northeast,0.1222,South,Northeast,True,"selection_rate, false_positive_rate"
3,employment_type,2,0.1431,Hourly,Salaried,0.1692,Hourly,Salaried,0.1402,Hourly,Salaried,True,"selection_rate, true_positive_rate, false_posi..."
4,department_name,8,0.2990,Customer Support,Information Technology,0.3972,Supply Chain,Information Technology,0.2883,Customer Support,Information Technology,True,"selection_rate, true_positive_rate, false_posi..."
5,job_level,3,0.1858,Level 1,Level 3,0.2430,Level 1,Level 3,0.1803,Level 1,Level 3,True,"selection_rate, true_positive_rate, false_posi..."
6,organizational_level,2,0.1875,Individual Contributor,Team Manager,0.1077,Individual Contributor,Team Manager,0.1949,Individual Contributor,Team Manager,True,"selection_rate, true_positive_rate, false_posi..."


## Validation and governance

The checks prove that the shared policy implementation reproduces Checkpoint 46, the proxy is explicitly different, no employee-level output is saved, and the frozen policy remains unchanged.

In [6]:
checks[["check", "status", "observed", "requirement"]]

,check,status,observed,requirement
0,Saved frozen policy is the configured expected...,PASS,budget_expected_value,budget_expected_value
1,Validation population and target contract are ...,PASS,"{'rows': 5521, 'positive_cases': 586}","{'rows': 5521, 'positive_cases': 586}"
2,Deployed and proxy capacities are independentl...,PASS,"{'deployed': 700, 'probability_proxy': 553}","{'deployed': 700, 'probability_proxy': 553}"
3,Different selection rules are explicitly demon...,PASS,184,184
4,Reconstructed selection reproduces Checkpoint 46,PASS,"{'selected_count': 700, 'selected_mean_salary_...","{'selected_count': 700, 'selected_mean_salary_..."
5,Expected selected salary is reproduced,PASS,121964.42857142857,121964.42857142857
6,Reported headline subgroup selections use depl...,PASS,"{'department_name': {'Customer Support': 5, 'E...","{'department_name': {'Customer Support': 5, 'E..."
7,Every configured policy subgroup is reported,PASS,"['age_band', 'department_name', 'education_lev...","['age_band', 'department_name', 'education_lev..."
8,All policy subgroup rates are valid,PASS,"{'minimum': 0.0, 'maximum': 0.7078651685393258}","All finite rates inside [0, 1]"
9,Reserved final-test target remains unread,PASS,"{'accesses_reserved_test_target': False, 'snap...","{'accesses_reserved_test_target': False, 'snap..."


### Portfolio navigation

Primary sequence: [Fairness, Economics, and Retention Policy](portfolio/03_fairness_economics_and_policy.ipynb)  
Related audit: [Retention Policy Allocation Equity](34_retention_policy_equity.ipynb)  
Notebook index: [README](README.md)